In [ ]:
# ---------------------------------------------------------------------------------
# Purpose: Create the star schema (GOLD Tier) tables.
# Accept parameters passed from orchestration notebook via dbutils.notebook.run()
# These simulate DAB variables in the bundle deployment 
# ---------------------------------------------------------------------------------

try:
    # Get parameters from dbutils.widgets (passed by dbutils.notebook.run)
    catalog_name = dbutils.widgets.get("catalog_name")
    schema_prefix = dbutils.widgets.get("schema_prefix")
    print(f"Using parameters from orchestration:")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")
except Exception:
    # Fallback to default values if not called from orchestration
    catalog_name = "dev_catalog"
    schema_prefix = "gld_star_hrs"
    print(f"Using default values (not called from orchestration):")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")

In [ ]:
# -----------------------------------------------------------------------------
# Initialize Notebook Configuration
# -----------------------------------------------------------------------------
# For Asset Bundles
#   Instead of relying on relative paths, add the bundle root to Python's path.
#   In each notebook that imports src, add this before the import:
#import sys
#sys.path.append("/Workspace/Users/peteperez.lv@gmail.com/.bundle/hrs_dbx_repo/default/files")

dbutils.widgets.dropdown(
    "truncate_table",
    "true",
    ["true", "false"]
)

# varibles for teting only
# catalog_name = "dev_catalog"
# schema_prefix = "gld_star_hrs"

#TRUNCATE_TABLE = dbutils.widgets.get("truncate_table").lower() == "true"

COHORT_TABLE = f"{catalog_name}.{schema_prefix}.dim_hrs_cohort"
WAVE_TABLE = f"{catalog_name}.{schema_prefix}.dim_hrs_wave"

LOAD_COHORT_SQL = "../../sql/gold_dml/load_dim_hrs_cohort_data.sql"
LOAD_WAVE_SQL = "../../sql/gold_dml/load_dim_hrs_wave_data.sql"

#VALIDATION_SQL = "../../sql/validation/validate_hrs_demographics_data.sql"

#SOURCE_TABLE = "dev_catalog.brz_raw_hrs.randhrs1992_2022v1"

In [ ]:
# Step 2
# Load distinct data to the COHORT_TABLE

from pathlib import Path

print("======================================================")
print("Step 2 - LOAD COHORT DATA")
print("======================================================")
try:
    # Read SQL file
    sql_path = Path(LOAD_COHORT_SQL)
    sql_text = sql_path.read_text()

    # Replace dev_catalog string with DAB catalog value.
    sql_text = sql_text.replace('dev_catalog', f"{catalog_name}")
    
    # Split and execute statements with parameter binding
    statements = [stmt.strip() for stmt in sql_text.split(';') if stmt.strip()]
    
    for i, stmt in enumerate(statements, 1):
        print(f"  Executing statement {i}/{len(statements)}")
        spark.sql(stmt, args={"catalog_name": catalog_name, "schema_prefix": schema_prefix})
    
    print("✓ Completed")
except Exception as e:
    print(f"❌ Load failed: {e}")
    raise


In [ ]:
# Step 3
# Load distinct data to the WAVE_TABLE

from pathlib import Path

print("======================================================")
print("Step 2 - LOAD wave DATA")
print("======================================================")
try:
    # Read SQL file
    sql_path = Path(LOAD_WAVE_SQL)
    sql_text = sql_path.read_text()

    # Replace dev_catalog string with DAB catalog value.
    sql_text = sql_text.replace('dev_catalog', f"{catalog_name}")
    
    # Split and execute statements with parameter binding
    statements = [stmt.strip() for stmt in sql_text.split(';') if stmt.strip()]
    
    for i, stmt in enumerate(statements, 1):
        print(f"  Executing statement {i}/{len(statements)}")
        spark.sql(stmt, args={"catalog_name": catalog_name, "schema_prefix": schema_prefix})
    
    print("✓ Completed")
except Exception as e:
    print(f"❌ Load failed: {e}")
    raise